In [2]:
from pathlib import Path
import pandas as pd
import torch


current_dir = Path.cwd()
if (current_dir / "data").exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = current_dir.parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "pigreid"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed" / "pigreid_224"
METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
MODELS_DIR = PROJECT_ROOT / "models"
RUNS_DIR = PROJECT_ROOT / "runs"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("RAW_DATA_DIR exists:", RAW_DATA_DIR.exists())
print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CUDA not available")

PROJECT_ROOT: d:\farm_project
RAW_DATA_DIR: d:\farm_project\data\raw\pigreid
RAW_DATA_DIR exists: True
torch version: 2.13.0+cu126
CUDA available: True
GPU: NVIDIA GeForce GTX 1660


In [3]:
all_items = list(RAW_DATA_DIR.iterdir())
identity_dirs = [item for item in all_items if item.is_dir()]
identity_dirs = sorted(identity_dirs, key=lambda path: path.name)

rows = []
for identity_dir in identity_dirs:
    identity_id = identity_dir.name
    image_paths = list(identity_dir.glob("*.png"))
    num_images = len(image_paths)
    rows.append({
        "identity_id": identity_id,
        "num_images": num_images,
        "folder_path": str(identity_dir)
    })

stats_df = pd.DataFrame(rows)
display(stats_df)
print("Total identities:", len(stats_df))
print("Total images:", stats_df["num_images"].sum())

,identity_id,num_images,folder_path
0,G12_54,1750,d:\farm_project\data\raw\pigreid\G12_54
1,G12_57,1750,d:\farm_project\data\raw\pigreid\G12_57
2,G12_61,1750,d:\farm_project\data\raw\pigreid\G12_61
3,G12_62,1750,d:\farm_project\data\raw\pigreid\G12_62
4,G12_66,1750,d:\farm_project\data\raw\pigreid\G12_66
5,G8_78,1250,d:\farm_project\data\raw\pigreid\G8_78
6,G8_79,1250,d:\farm_project\data\raw\pigreid\G8_79
7,G8_82,1250,d:\farm_project\data\raw\pigreid\G8_82
8,G8_86,1250,d:\farm_project\data\raw\pigreid\G8_86
9,G8_87,1250,d:\farm_project\data\raw\pigreid\G8_87


Total identities: 12
Total images: 17500


In [4]:
NUM_IDENTITIES = 6
IMAGES_PER_IDENTITY = 80

stats_sorted = stats_df.sort_values("num_images", ascending=False).reset_index(drop=True)
selected_identities = stats_sorted.head(NUM_IDENTITIES)["identity_id"].tolist()
print("Selected identities:", selected_identities)

subset_rows = []

for identity_id in selected_identities:
    identity_dir = RAW_DATA_DIR / identity_id
    image_paths = sorted(identity_dir.glob("*.png"))
    if len(image_paths) < IMAGES_PER_IDENTITY:
        raise ValueError(f"У {identity_id} только {len(image_paths)} изображений, а нужно {IMAGES_PER_IDENTITY}.")

    selected_indices = [round(i * (len(image_paths) - 1) / (IMAGES_PER_IDENTITY - 1)) for i in range(IMAGES_PER_IDENTITY)]

    selected_images = [image_paths[index] for index in selected_indices]

    for image_path in selected_images:
        subset_rows.append({
            "image_path": str(image_path),
            "identity_id": identity_id,
            "file_name": image_path.name
        })

subset_df = pd.DataFrame(subset_rows)
display(subset_df.head())
print("Subset size:", len(subset_df))
display(subset_df["identity_id"].value_counts())

Selected identities: ['G12_54', 'G12_57', 'G12_61', 'G12_62', 'G12_66', 'G8_78']


,image_path,identity_id,file_name
0,d:\farm_project\data\raw\pigreid\G12_54\Basler...,G12_54,Basler_acA4112-20uc__40331001__20240716_130747...
1,d:\farm_project\data\raw\pigreid\G12_54\Basler...,G12_54,Basler_acA4112-20uc__40331001__20240716_130747...
2,d:\farm_project\data\raw\pigreid\G12_54\Basler...,G12_54,Basler_acA4112-20uc__40331001__20240716_130747...
3,d:\farm_project\data\raw\pigreid\G12_54\Basler...,G12_54,Basler_acA4112-20uc__40331001__20240716_130747...
4,d:\farm_project\data\raw\pigreid\G12_54\Basler...,G12_54,Basler_acA4112-20uc__40331001__20240716_130747...


Subset size: 480


identity_id
G12_54    80
G12_57    80
G12_61    80
G12_62    80
G12_66    80
G8_78     80
Name: count, dtype: int64

In [5]:
from PIL import Image
from tqdm.auto import tqdm

IMAGE_SIZE = 224

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

processed_rows = []

for row in tqdm(subset_df.itertuples(index=False), total=len(subset_df)):
    
    src_path = Path(row.image_path)
    identity_id = row.identity_id
    dst_identity_dir = PROCESSED_DATA_DIR / identity_id
    dst_identity_dir.mkdir(parents=True, exist_ok=True)
    dst_file_name = src_path.stem + ".jpg"
    dst_path = dst_identity_dir / dst_file_name
    image = Image.open(src_path)
    image = image.convert("RGB")

    image = image.resize((IMAGE_SIZE, IMAGE_SIZE))
    image.save(dst_path, quality=95)

    processed_rows.append({
        "image_path": str(dst_path),
        "identity_id": identity_id,
        "original_path": str(src_path),
        "file_name": dst_file_name
    })

metadata_df = pd.DataFrame(processed_rows)

display(metadata_df.head())

print("Processed images:", len(metadata_df))
display(metadata_df["identity_id"].value_counts())

  0%|          | 0/480 [00:00<?, ?it/s]

,image_path,identity_id,original_path,file_name
0,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,d:\farm_project\data\raw\pigreid\G12_54\Basler...,Basler_acA4112-20uc__40331001__20240716_130747...
1,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,d:\farm_project\data\raw\pigreid\G12_54\Basler...,Basler_acA4112-20uc__40331001__20240716_130747...
2,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,d:\farm_project\data\raw\pigreid\G12_54\Basler...,Basler_acA4112-20uc__40331001__20240716_130747...
3,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,d:\farm_project\data\raw\pigreid\G12_54\Basler...,Basler_acA4112-20uc__40331001__20240716_130747...
4,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,d:\farm_project\data\raw\pigreid\G12_54\Basler...,Basler_acA4112-20uc__40331001__20240716_130747...


Processed images: 480


identity_id
G12_54    80
G12_57    80
G12_61    80
G12_62    80
G12_66    80
G8_78     80
Name: count, dtype: int64

In [6]:
METADATA_DIR.mkdir(parents=True, exist_ok=True)
metadata_path = METADATA_DIR / "metadata.csv"
metadata_df.to_csv(metadata_path, index=False)
print("Saved metadata to:", metadata_path)

Saved metadata to: d:\farm_project\data\metadata\metadata.csv


In [7]:
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
TEST_SIZE = 0.20

QUERY_IMAGES_PER_ID = 4
INITIAL_LABELED_FRACTION = 0.20

split_rows = []

for identity_id, group_df in metadata_df.groupby("identity_id"):

    group_df = group_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    train_df, test_df = train_test_split(
        group_df,
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED,
        shuffle=True
    )

    test_df = test_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    query_df = test_df.iloc[:QUERY_IMAGES_PER_ID].copy()
    gallery_df = test_df.iloc[QUERY_IMAGES_PER_ID:].copy()
    train_df = train_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    n_initial_labeled = max(2, int(len(train_df) * INITIAL_LABELED_FRACTION))
    initial_labeled_df = train_df.iloc[:n_initial_labeled].copy()
    unlabeled_df = train_df.iloc[n_initial_labeled:].copy()
    
    initial_labeled_df["split"] = "train_pool"
    initial_labeled_df["pool_status"] = "initial_labeled"
    initial_labeled_df["is_labeled"] = True
    unlabeled_df["split"] = "train_pool"
    unlabeled_df["pool_status"] = "unlabeled"
    unlabeled_df["is_labeled"] = False
    query_df["split"] = "test"
    query_df["pool_status"] = "query"
    query_df["is_labeled"] = True
    gallery_df["split"] = "test"
    gallery_df["pool_status"] = "gallery"
    gallery_df["is_labeled"] = True
    split_rows.append(initial_labeled_df)
    split_rows.append(unlabeled_df)
    split_rows.append(query_df)
    split_rows.append(gallery_df)

split_df = pd.concat(split_rows, ignore_index=True)
display(split_df.head())
print("Total rows:", len(split_df))
display(split_df["split"].value_counts())
display(split_df["pool_status"].value_counts())
display(pd.crosstab(split_df["identity_id"], split_df["pool_status"]))

,image_path,identity_id,original_path,file_name,split,pool_status,is_labeled
0,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,d:\farm_project\data\raw\pigreid\G12_54\Basler...,Basler_acA4112-20uc__40331001__20240806_074832...,train_pool,initial_labeled,True
1,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,d:\farm_project\data\raw\pigreid\G12_54\Basler...,Basler_acA4112-20uc__40331001__20240826_101022...,train_pool,initial_labeled,True
2,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,d:\farm_project\data\raw\pigreid\G12_54\Basler...,Basler_acA4112-20uc__40331001__20240729_124640...,train_pool,initial_labeled,True
3,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,d:\farm_project\data\raw\pigreid\G12_54\Basler...,Basler_acA4112-20uc__40331001__20240729_124640...,train_pool,initial_labeled,True
4,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,d:\farm_project\data\raw\pigreid\G12_54\Basler...,Basler_acA4112-20uc__40331001__20240820_083924...,train_pool,initial_labeled,True


Total rows: 480


split
train_pool    384
test           96
Name: count, dtype: int64

pool_status
unlabeled          312
initial_labeled     72
gallery             72
query               24
Name: count, dtype: int64

pool_status,gallery,initial_labeled,query,unlabeled
identity_id,,,,
G12_54,12,12,4,52
G12_57,12,12,4,52
G12_61,12,12,4,52
G12_62,12,12,4,52
G12_66,12,12,4,52
G8_78,12,12,4,52


In [8]:
METADATA_DIR.mkdir(parents=True, exist_ok=True)
splits_path = METADATA_DIR / "splits.csv"
split_df.to_csv(splits_path, index=False)
print("Saved splits to:", splits_path)

Saved splits to: d:\farm_project\data\metadata\splits.csv


In [9]:
file_exists = split_df["image_path"].apply(lambda path: Path(path).exists())
print("Existing files:", file_exists.sum())
print("Total files:", len(file_exists))
if file_exists.sum() != len(file_exists):
    display(split_df.loc[~file_exists].head())
else:
    print("All files exist.")

Existing files: 480
Total files: 480
All files exist.
